# Module 2 Exercise: EXPLAIN EXTENDED and the Four Catalyst Stages

SQL Server shows one plan (estimated or actual). Catalyst shows four: Parsed, Analyzed,
Optimized, Physical. `EXPLAIN EXTENDED` prints all of them so you can see *where* each
rewrite happened.


In [ ]:
%%sql
EXPLAIN EXTENDED
SELECT (quantity + 1 + 2 + 3) AS quantity FROM silver.dbo.orders
WHERE year = 2025
LIMIT 1000

## What to look for

Scroll the result from top to bottom. Four sections separated by `== ... ==` headers.

- **Parsed Logical Plan**: your SQL as-written. `((quantity + 1) + 2) + 3` is still a tree.
- **Analyzed Logical Plan**: table and column references resolved against the catalog.
- **Optimized Logical Plan**: constant folding collapses the arithmetic to `quantity + 6`,
  and `year = 2025` becomes a partition filter.
- **Physical Plan**: the operators Spark will actually run. Look for `PartitionFilters`
  on the Delta scan, that is partition pruning at work.


## Key takeaway

The `Parsed` → `Optimized` diff is the one that matters. If the plan you wrote and the
plan Catalyst chose look the same, no rewrite happened. If they differ, Catalyst saved
you work: the fewer operators downstream, the less data the executors touch.
